In [ ]:
import sys
import os

# Add paths to import from long_form_factuality
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
lff_root = os.path.join(project_root, "long_form_factuality")
for path in [project_root, lff_root]:
    if path not in sys.path:
        sys.path.insert(0, path)

from vllm_wrapper import VLLMAtomizationModel

# Import the original SAFE implementation
from third_party.factscore import atomic_facts
import itertools

def get_atomic_facts_safe(response: str, model, debug=False, query=None):
    """Wrapper that uses the original SAFE implementation with correct paths."""
    demon_dir = os.path.join(lff_root, "third_party", "factscore", "demos")
    atomic_fact_generator = atomic_facts.AtomicFactGenerator(
        api_key='', 
        demon_dir=demon_dir,
        gpt3_cache_file='', 
        other_lm=model,
        query=query
    )
    
    # Monkey patch the generate method to print prompts if debug=True
    if debug:
        original_generate = model.generate
        def debug_generate(prompt, **kwargs):
            print("="*80)
            print("PROMPT SENT TO MODEL:")
            print("="*80)
            print(prompt)
            print("="*80)
            result = original_generate(prompt, **kwargs)
            print("\nMODEL RESPONSE:")
            print("="*80)
            print(result)
            print("="*80)
            return result
        model.generate = debug_generate
    
    facts, _ = atomic_fact_generator.run(response)
    
    # Restore original generate if we patched it
    if debug:
        model.generate = original_generate
    
    # Convert to dict format
    facts_as_dict = [
        {'sentence': sentence, 'atomic_facts': identified_atomic_facts}
        for sentence, identified_atomic_facts in facts
    ]
    all_atomic_facts_list = list(
        itertools.chain.from_iterable([f['atomic_facts'] for f in facts_as_dict])
    )
    
    return {
        'num_claims': len(all_atomic_facts_list),
        'sentences_and_atomic_facts': facts,
        'all_atomic_facts': facts_as_dict,
    }


In [6]:
dummy_text_to_atomize = "He had an IQ of 160"
llm = VLLMAtomizationModel()
atomized = get_atomic_facts_safe(dummy_text_to_atomize, llm, debug=False, query="Tell me about Albert Einstein")

print(atomized)

{'num_claims': 8, 'sentences_and_atomic_facts': [('He had an IQ of 160', ['Albert Einstein was born on March 14, 1879.', 'Albert Einstein was a German-born theoretical physicist.', 'Albert Einstein developed the theory of relativity.', 'Albert Einstein received the Nobel Prize in Physics in 1921.', 'Albert Einstein published the equation E=mc².', 'Albert Einstein immigrated to the United States in 1933.', 'Albert Einstein worked at the Institute for Advanced Study in Princeton.', 'Albert Einstein died on April 18, 1955.'])], 'all_atomic_facts': [{'sentence': 'He had an IQ of 160', 'atomic_facts': ['Albert Einstein was born on March 14, 1879.', 'Albert Einstein was a German-born theoretical physicist.', 'Albert Einstein developed the theory of relativity.', 'Albert Einstein received the Nobel Prize in Physics in 1921.', 'Albert Einstein published the equation E=mc².', 'Albert Einstein immigrated to the United States in 1933.', 'Albert Einstein worked at the Institute for Advanced Study 

In [7]:
import json
import os

responses = []
with open(os.getcwd() + "/data_for_git/responses.jsonl", "r") as f:
    for line in f:
        responses.append(json.loads(line))

# print one of the responses

def get_original_prompt(prompt):
    return prompt.split("Based on the documents above, i now want you to: ")[1].split(",")[0]

In [8]:
import threading
from tqdm import tqdm

max_concurrent_requests = 50

request_semaphore = threading.Semaphore(max_concurrent_requests)

# Thread-safe list to collect results
results = []
results_lock = threading.Lock()
error_log = []
error_log_lock = threading.Lock()
def worker(prompt, response, model, pbar):
    try:
        request_semaphore.acquire()
        original_prompt = get_original_prompt(prompt)
        """Worker function that collects results"""
        result = get_atomic_facts_safe(response["response"], model, debug=False, query=original_prompt)
        # print("original prompt", original_prompt)
        # print("prompt", prompt)
        with results_lock:
            results.append({"id": response["id"], "prompt": original_prompt, "response": response["response"], "results": result})
            pbar.update(1)
    except Exception as e:
        print(f"Error processing response {response['id']}: {e}")
        with error_log_lock:
            error_log.append(f"Error processing response {response['id']}: {e}")
    finally:
        request_semaphore.release()

first_n = None

total_tasks = sum(len(q["responses"]) for q in responses[:first_n])

threads = []
# Fix: enumerate returns (index, item), so iterate directly
with tqdm(total=total_tasks, desc="Proce    ssing responses") as pbar:
    for query in responses[:first_n]:
        for response in query["responses"]:
            t = threading.Thread(target=worker, args=(query["prompt"], response, llm, pbar))
            t.start()
            threads.append(t)

    for t in threads:
        t.join()

with open(os.getcwd() + "/data_for_git/atomic_facts_error_log.txt", "w") as f:
    for error in error_log:
        f.write(error + "\n")

# Now results contains all the atomic facts
print(f"Processed {len(results)} responses")
# Access results: results[0], results[1], etc.

Proce    ssing responses:  10%|▉         | 336/3415 [09:51<1:30:22,  1.76s/it]


KeyboardInterrupt: 

In [ ]:
# check that the number of sentences in the atomization is the same as the number of sentences in logprobs
# print(responses[0]["responses"])
# match result id:s with response id:s
matches = 0
for result in results:
    for query_responses in responses:
        for response in query_responses["responses"]:
            if result['id'] == response['id']:
                if len(result['results']['sentences_and_atomic_facts']) != len(response['logprobs']):
                    print(f"Number of sentences in atomization ({len(result['results']['sentences_and_atomic_facts'])}) does not match number of sentences in logprobs ({len(response['logprobs'])}) for response {result['id']}")
                else:
                    matches += 1
                    print(f"Number of sentences in atomization ({len(result['results']['sentences_and_atomic_facts'])}) matches number of sentences in logprobs ({len(response['logprobs'])}) for response {result['id']}")
print(f"Matches: {matches}")

Number of sentences in atomization (2) matches number of sentences in logprobs (2) for response 598f763a-d239-4b4b-8f55-a04164f6d223
Number of sentences in atomization (3) matches number of sentences in logprobs (3) for response d357548d-9c1c-478f-a5ea-3784abc534c5
Number of sentences in atomization (2) matches number of sentences in logprobs (2) for response 530934e1-887a-4aa4-8423-e1c44efb770a
Number of sentences in atomization (3) matches number of sentences in logprobs (3) for response aeb78867-09db-40d1-86b5-124cae3bae6e
Number of sentences in atomization (2) matches number of sentences in logprobs (2) for response 09d40c95-c821-4f75-b86b-3dfdf4e4c5c2
Number of sentences in atomization (2) matches number of sentences in logprobs (2) for response c7e803cb-af4e-4344-9b13-e3bc0d07940b
Number of sentences in atomization (3) matches number of sentences in logprobs (3) for response 2eea7425-c1bf-4a69-9aa5-dede640f5a20
Number of sentences in atomization (2) matches number of sentences in

In [ ]:
import json
with open(os.getcwd() + "/data_for_git/atomic_facts.jsonl", "w") as f:
    for result in results:
        f.write(json.dumps(result) + "\n")
